# Part 4d — Stackelberg leadership as a single-level MPEC

### Collapsing a bilevel program using the follower's KKT conditions

In Cournot (4c) both firms move simultaneously. Here the **leader commits first** and the follower
observes that commitment before choosing its own output and capacity. That is a **bilevel** program:

$$\max_{x_L} \;\; \Pi_L(x_L, x_F^*) \quad \text{s.t.} \quad
x_F^* \in \arg\max_{x_F} \Pi_F(x_F; x_L)$$

An optimisation nested inside an optimisation cannot be handed to a solver directly. The standard
route is to replace the inner problem with its **optimality conditions**, producing a single-level
**MPEC** (Mathematical Program with Equilibrium Constraints).

**This is only possible because the follower's problem is continuous and concave.** KKT conditions
are necessary and sufficient for a concave program; they say nothing useful about a MILP. So the
design decision made all the way back in Part 3 — keep the operational layer an LP, put the
integers only in investment — is what makes this notebook possible at all.

### The division of labour

| | Leader (R1) | Follower (R2) |
|---|---|---|
| Investment | binary build + continuous size, full chain | **continuous** capacity expansion |
| Output | chosen on a discrete grid | continuous |
| Problem type | MILP | **concave QP** → KKT applies |

The follower's simplification is a real modelling cost and worth stating plainly: it can expand
capacity continuously but cannot make lumpy siting decisions. Its investment is a scalar, not a
plan. That is the price of admission for an exact bilevel formulation.

## The formulation

**Follower's problem**, taking the leader's quantities $q^L$ as given:

$$\max_{q^F \ge 0,\, \kappa \ge 0} \;\; \sum_{rt,p} \omega_p\Big[\big(A_{rt,p} - B_{rt,p}(q^F_{rt,p}+q^L_{rt,p}) - c_{rt}\big)q^F_{rt,p}\Big] - K\kappa$$
$$\text{s.t.} \quad \sum_{rt} q^F_{rt,p} \;\le\; \text{legacy}_p + \kappa \quad [\lambda_p]$$

**KKT conditions**, which become constraints in the leader's model:

*Stationarity*
$$\omega_p\big(A_{rt,p} - B_{rt,p}(2q^F_{rt,p} + q^L_{rt,p}) - c_{rt}\big) - \lambda_p + \nu_{rt,p} = 0$$
$$-K + \sum_p \lambda_p + \mu_\kappa = 0$$

*Complementarity*, linearised with binaries and big-M:
$$\lambda_p \le M y_p, \quad \text{slack}_p \le M(1-y_p)$$
$$\nu_{rt,p} \le M z_{rt,p}, \quad q^F_{rt,p} \le M(1-z_{rt,p})$$

Note the factor of **2** on $q^F$ in stationarity — that is the follower internalising its own price
impact, and it is the mathematical signature of Cournot behaviour.

### The bilinear obstacle, and an exact way around it

The leader's revenue is

$$\big(A - B(q^L + q^F)\big)q^L \;=\; A q^L - B (q^L)^2 - B\, \underbrace{q^F q^L}_{\text{bilinear}}$$

That last term is a product of two *decision variables* — nonconvex, and not reachable by the
piecewise trick from 4c because there the rival's quantity was a fixed parameter.

**Solution: put the leader's quantity on a binary-selected grid.**
$q^L = \sum_k S_k\, b_k$ with $\sum_k b_k = 1$ and $b_k$ binary. Then

$$q^F q^L = \sum_k S_k\,(q^F b_k)$$

and each $q^F b_k$ is *continuous × binary*, which linearises **exactly**:

$$w_k \le M b_k, \qquad w_k \le q^F, \qquad w_k \ge q^F - M(1-b_k), \qquad w_k \ge 0$$

No relaxation, no McCormick gap. The only approximation is that the leader's quantity is restricted
to a grid — a discretised strategy space, which is standard for MPECs and whose fineness is a
parameter you can sweep.

## 1. Setup and shared core

In [ ]:
!pip install gurobipy --quiet
import os, math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})
ENV = None          # set to gp.Env(params=...) to use a full WLS licence
print("gurobipy", gp.gurobi.version())

In [ ]:
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']

# ---------------- TIME ----------------
import os
# P4_SMALL shrinks the horizon so the EXACT MIQP fits the size-limited licence,
# letting us validate the piecewise-linear revenue against the true quadratic.
BLOCKS = ([(2, 1), (1, 3)] if os.environ.get('P4_SMALL') else [(6, 1), (4, 3), (2, 5), (1, 9)])
LEN, START = [], []
_y = 1
for _c, _L in BLOCKS:
    for _ in range(_c):
        LEN.append(_L); START.append(_y); _y += _L
P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
DR = 0.05
OMEGA = {p: sum(1/(1+DR)**t for t in YEARS[p]) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}
REPORT_UNTIL = 28

# ---------------- TECH ----------------
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0
CRF = DR*(1+DR)**LIFE/((1+DR)**LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF*sum(1/(1+DR)**t for t in range(ONLINE[s, v], ONLINE[s, v]+LIFE)
                      if t <= HORIZON) for s in STAGES for v in P}

In [ ]:
# ---------------- ASYMMETRIC INSTANCE ----------------
# R1 = incumbent upstream processor with accumulated experience.
# R2 = entrant, cheaper to build, trying to move downstream.
FIXED = {('MINE','R1'):900.,('PROC','R1'):1500.,('MFG','R1'):1300.,
         ('MINE','R2'):820.,('PROC','R2'):1350.,('MFG','R2'):1180.}
UNIT  = {('MINE','R1'):7.0,('PROC','R1'):11.0,('MFG','R1'):9.5,
         ('MINE','R2'):6.4,('PROC','R2'):10.0,('MFG','R2'):8.7}
OPEX  = {('MINE','R1'):1.2,('PROC','R1'):2.0,('MFG','R1'):2.4,
         ('MINE','R2'):1.35,('PROC','R2'):2.2,('MFG','R2'):2.6}

LEGACY_CAP = {('MINE','R1'):230,('PROC','R1'):205,('MFG','R1'):155,
              ('MINE','R2'):165,('PROC','R2'):125,('MFG','R2'):100}
LEGACY_RET = {('MINE','R1'):11,('PROC','R1'):14,('MFG','R1'):18,
              ('MINE','R2'):9, ('PROC','R2'):16,('MFG','R2'):22}
LEGACY_BYR = -8
# incumbent starts with accumulated production experience
EXPERIENCE0 = {'R1': 2600.0, 'R2': 500.0}

In [ ]:
# ---------------- EFFICIENCY (yield) ----------------
ETA_CEIL = {'MINE':0.92,'PROC':0.95,'MFG':0.93}
ETA_BASE = {'MINE':0.86,'PROC':0.80,'MFG':0.78}
ALPHA    = {'MINE':0.0,'PROC':0.030,'MFG':0.025}
BETA     = {'MINE':0.0,'PROC':0.010,'MFG':0.008}
DELTA_BAR= {'MINE':0.02,'PROC':0.05,'MFG':0.05}
ETA_FLOOR= 0.60
VINTAGES = [-1] + P
BYEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}
ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s]-ETA_BASE[s])*(1-ALPHA[s])**(BYEAR[v]-1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p]-BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s]-fr)*(1-BETA[s])**age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr+DELTA_BAR[s], aged))

In [ ]:
# ---------------- DEMAND & MARKET ----------------
DEMAND = {}
for r, base, g in [('R1', 100.0, 0.008), ('R2', 75.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base*(1+g)**(t-1) for t in YEARS[p])/LEN[p]
TRANSPORT = {(rf, rt): (0.5 if rf == rt else 2.4) for rf in REGIONS for rt in REGIONS}
PRICE_FIXED = 12.0
PEN_SHORT, PEN_DISPOSE = 90.0, 12.0

In [ ]:
# ---------------- LEARNING ----------------
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX, Q_START, Q_ADD, CAPEX_FLOOR, NBP = 0.15, 300.0, 700.0, 0.60, 9
_bc = -math.log2(1-LR_CAPEX)
K = list(range(NBP))
QBP = [Q_START + Q_ADD*k/(NBP-1) for k in K]

def _cap_unit_mult(q):
    return max(CAPEX_FLOOR, (q/Q_START)**(-_bc))

def _cap_cum_mult(q, n=400):
    if q <= Q_START:
        return 0.0
    h = (q-Q_START)/n
    return sum(0.5*(_cap_unit_mult(Q_START+i*h)+_cap_unit_mult(Q_START+(i+1)*h))*h
               for i in range(n))
CBP = [_cap_cum_mult(q) for q in QBP]

LR_OPEX, OPEX_FLOOR, LAG_YEARS, N_TIERS = 0.18, 0.65, 3, 3
TIER_Q, TIER_M = {}, {}

def set_tiers(top_by_region):
    for r in REGIONS:
        top = max(top_by_region[r], 1.0)
        q1 = top/8.0
        TIER_Q[r] = [q1*2**j for j in range(N_TIERS-1)]
        TIER_M[r] = [max(OPEX_FLOOR, (1-LR_OPEX)**j) for j in range(N_TIERS)]

ACTIVE = {r: [(s, v, p) for s in STAGES for v in VINTAGES for p in P
              if (v == -1 and START[p] <= LEGACY_RET[s, r])
              or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v]+LIFE-1)]
          for r in REGIONS}
VIN = {(r, s, p): [v for (ss, v, pp) in ACTIVE[r] if (ss, pp) == (s, p)]
       for r in REGIONS for s in STAGES for p in P}
BUILD = {r: [(s, v) for s in STAGES for v in P if ONLINE[s, v] <= HORIZON]
         for r in REGIONS}

In [ ]:
def add_region(m, r, learning='both'):
    """Attach one region's vertically-integrated chain to model m. Returns handles."""
    b = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')
    c = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')
    x = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')
    f_mp = m.addVars(P, lb=0.0, name=f'fmp_{r}')
    f_pf = m.addVars(P, lb=0.0, name=f'fpf_{r}')
    sale = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')
    disp = m.addVars(P, lb=0.0, name=f'disp_{r}')

    m.addConstrs((c[s, v] <= CAP_MAX*b[s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c[s, v] >= CAP_MIN*b[s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x[s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c[s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')
    m.addConstrs((gp.quicksum(ETA['MINE', v, p]*x['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == f_mp[p] for p in P),
                 name=f'mine_{r}')
    m.addConstrs((f_mp[p] == gp.quicksum(x['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p]*x['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == f_pf[p] for p in P),
                 name=f'pout_{r}')
    m.addConstrs((f_pf[p] == gp.quicksum(x['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p]*x['MFG', v, p]
                              for v in VIN[r, 'MFG', p])
                  == sale.sum('*', p) + disp[p] for p in P), name=f'mout_{r}')

    # cumulative production (undiscounted), regional scope, with initial experience
    cum = m.addVars(P, lb=0.0, ub=3*CAP_MAX*HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum[p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q]*x['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

    capex = gp.quicksum(MU[s, v]*FIXED[s, r]*b[s, v] for (s, v) in BUILD[r]) \
          + gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                        for (s, v) in BUILD[r] if s not in LEARN_STAGES)
    if learning in ('capacity', 'both'):
        Q = m.addVars(P, lb=Q_START, ub=Q_START+Q_ADD, name=f'Q_{r}')
        Cc = m.addVars(P, lb=0.0, name=f'C_{r}')
        lam = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
        m.addConstrs((lam.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
        m.addConstrs((Q[p] == gp.quicksum(QBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sQ_{r}')
        m.addConstrs((Cc[p] == gp.quicksum(CBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sC_{r}')
        m.addConstrs((Q[p] == Q_START + gp.quicksum(c[s, v] for (s, v) in BUILD[r]
                                                    if s in LEARN_STAGES and v <= p)
                      for p in P), name=f'cc_{r}')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])
        rate = sum(UNIT[s, r] for s in LEARN_STAGES)/len(LEARN_STAGES)
        capex += gp.quicksum(MU['PROC', p]*rate*(Cc[p]-(Cc[p-1] if p > 0 else 0.0))
                             for p in P)
    else:
        capex += gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                             for (s, v) in BUILD[r] if s in LEARN_STAGES)

    if learning in ('production', 'both') and TIER_Q:
        J = list(range(N_TIERS))
        z = m.addVars(P, J, vtype=GRB.BINARY, name=f'z_{r}')
        m.addConstrs((z.sum(p, '*') == 1 for p in P), name=f'ot_{r}')
        LAGP = {p: YEAR_TO_P[max(1, START[p]-LAG_YEARS)] for p in P}
        BIGQ = 3*CAP_MAX*HORIZON + EXPERIENCE0[r]
        m.addConstrs((cum[LAGP[p]] >= TIER_Q[r][j-1] - BIGQ*(1-z[p, j])
                      for p in P for j in J if j > 0), name=f'tf_{r}')
        m.addConstrs((cum[LAGP[p]] <= TIER_Q[r][j] + BIGQ*(1-z[p, j])
                      for p in P for j in J if j < N_TIERS-1), name=f'tc_{r}')
        ts = m.addVars(STAGES, P, J, lb=0.0, name=f'ts_{r}')
        m.addConstrs((ts.sum(s, p, '*') == gp.quicksum(x[s, v, p] for v in VIN[r, s, p])
                      for s in STAGES for p in P), name=f'tss_{r}')
        m.addConstrs((ts[s, p, j] <= 3*CAP_MAX*z[p, j]
                      for s in STAGES for p in P for j in J), name=f'tl_{r}')
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*TIER_M[r][j]*ts[s, p, j]
                           for s in STAGES for p in P for j in J)
    else:
        z = None
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*x[s, v, p] for (s, v, p) in ACTIVE[r])

    trans = gp.quicksum(OMEGA[p]*TRANSPORT[r, rt]*sale[rt, p] for rt in REGIONS for p in P)
    dcost = gp.quicksum(OMEGA[p]*PEN_DISPOSE*disp[p] for p in P)
    revenue = gp.quicksum(OMEGA[p]*PRICE_FIXED*sale[rt, p] for rt in REGIONS for p in P)
    return dict(b=b, c=c, x=x, sale=sale, disp=disp, cum=cum, z=z,
                capex=capex, opex=opex, trans=trans, dcost=dcost, revenue=revenue,
                cost=capex+opex+trans+dcost)

In [ ]:
def solve_planner(w1=0.5, learning='both', mipgap=0.005, quiet=True):
    m = gp.Model(); m.Params.OutputFlag = 0 if quiet else 1; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    short = m.addVars(REGIONS, P, lb=0.0, name='short')
    m.addConstrs((gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS) + short[rt, p]
                  >= DEMAND[rt, p] for rt in REGIONS for p in P), name='demand')
    pen = gp.quicksum(OMEGA[p]*PEN_SHORT*short[rt, p] for rt in REGIONS for p in P)
    m.setObjective(w1*H['R1']['cost'] + (1-w1)*H['R2']['cost'] + pen, GRB.MINIMIZE)
    m.optimize()
    m._H, m._short, m._pen = H, short, pen
    return m

In [ ]:
# ================= 4c: Cournot with endogenous price =================
CHOKE    = 30.0     # price at zero quantity
P_ANCHOR = 13.0     # price when quantity equals the Part 4b demand reference
A_INT = {(rt, p): CHOKE for rt in REGIONS for p in P}
B_SLP = {(rt, p): (CHOKE - P_ANCHOR) / DEMAND[rt, p] for rt in REGIONS for p in P}


NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
def best_response_cournot(r, rival_sales, learning='both', mipgap=0.005):
    """Firm r maximises profit facing linear inverse demand
       p[rt,p] = A - B*(own + rival).
    Revenue is piecewise-linearised in own quantity, keeping the model a MILP."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    h = add_region(m, r, learning)
    s = h['sale']
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            q_bar = rival_sales.get((rt, p), 0.0)
            a_eff = A_INT[rt, p] - B_SLP[rt, p] * q_bar
            smax = max(1e-6, A_INT[rt, p] / B_SLP[rt, p] - q_bar)
            S, R = _rev_breakpoints(a_eff, B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1, name=f'rcvx_{rt}_{p}')
            m.addConstr(s[rt, p] == gp.quicksum(S[k] * mu[rt, p, k] for k in KR),
                        name=f'rS_{rt}_{p}')
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR),
                        name=f'rR_{rt}_{p}')
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h, revenue
    return m

In [ ]:
def cournot_iterate(learning='both', first='R1', max_iter=16, tol=0.5, mipgap=1e-3):
    """Iterated best response under Cournot competition.

    Convergence for a game with CONTINUOUS strategies must be tested with a
    TOLERANCE, not by exact state matching: each best response is a MILP solved to
    a finite gap, so the returned quantities wobble slightly between iterations.
    Exact hashing reads that wobble as a cycle."""
    def dist(a, b):
        return max(abs(a[r][k] - b[r][k]) for r in REGIONS for k in a[r])

    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    plans, hist, log = {}, [], []
    order = [first, 'R2' if first == 'R1' else 'R1']
    for it in range(max_iter):
        prev = {r: dict(sales[r]) for r in REGIONS}
        for r in order:
            other = 'R2' if r == 'R1' else 'R1'
            m = best_response_cournot(r, sales[other], learning=learning, mipgap=mipgap)
            if m.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = {(rt, p): m._h['sale'][rt, p].X for rt in REGIONS for p in P}
            plans[r] = tuple(sorted((s_, v) for (s_, v) in m._h['b']
                                    if m._h['b'][s_, v].X > 0.5))
            log.append(dict(iter=it, firm=r, profit=m.ObjVal,
                            revenue=m._rev.getValue(), cost=m._h['cost'].getValue(),
                            builds=len(plans[r]), sales=sum(sales[r].values()),
                            disposal=sum(m._h['disp'][p].X for p in P)))
        cur = {r: dict(sales[r]) for r in REGIONS}
        if it > 0 and dist(cur, prev) < tol:
            return dict(status='CONVERGED', cycle_len=1, iters=it + 1, log=log,
                        plans=plans, sales=sales, drift=dist(cur, prev))
        for k, past in enumerate(hist):                      # genuine k-cycle, k >= 2
            if dist(cur, past) < tol:
                return dict(status='CYCLE', cycle_len=len(hist) - k, iters=it + 1,
                            log=log, plans=plans, sales=sales)
        hist.append(cur)
    return dict(status='MAX_ITER', iters=max_iter, log=log, plans=plans, sales=sales)

In [ ]:
def market_outcome(sales):
    rows = []
    for rt in REGIONS:
        for p in P:
            q = sum(sales[r][rt, p] for r in REGIONS)
            price = A_INT[rt, p] - B_SLP[rt, p] * q
            rows.append(dict(market=rt, period=p, year=START[p], quantity=q, price=price,
                             consumer_surplus=0.5 * B_SLP[rt, p] * q * q,
                             share_R1=(sales['R1'][rt, p] / q if q > 1e-6 else None)))
    return rows

## 2. The follower's cost and inherited capacity

The follower's marginal cost per unit **delivered** must account for chain yields: to deliver one
finished unit you need $1/\eta_{MFG}$ of manufacturing throughput, $1/(\eta_{MFG}\eta_{PROC})$ of
processing, and so on upstream. Transport to each market is added on top, which is why serving the
leader's home market costs the follower more.

In [ ]:
FOLLOWER, LEADER = 'R2', 'R1'

# --- follower's marginal cost per unit DELIVERED to each market -------------
def follower_marginal_cost():
    e_m, e_p, e_f = ETA['MINE', -1, 0], ETA['PROC', -1, 0], ETA['MFG', -1, 0]
    thr_f = 1.0/e_f
    thr_p = thr_f/e_p
    thr_m = thr_p/e_m
    chain = (OPEX['MFG', FOLLOWER]*thr_f + OPEX['PROC', FOLLOWER]*thr_p
             + OPEX['MINE', FOLLOWER]*thr_m)
    return {rt: chain + TRANSPORT[FOLLOWER, rt] for rt in REGIONS}

# follower's inherited deliverable capacity, and the annualised cost of adding more
def follower_legacy(p):
    return LEGACY_CAP['MFG', FOLLOWER]*ETA['MFG', -1, p] if START[p] <= LEGACY_RET['MFG', FOLLOWER] else 0.0

CAP_COST = sum(MU['MFG', v] for v in P)/len(P)*(UNIT['MFG', FOLLOWER]*1.0) + 4.0
BIG_Q, BIG_L = 1200.0, 400.0

In [ ]:
c_f = follower_marginal_cost()
print("follower marginal cost per unit delivered:", {k: round(v, 3) for k, v in c_f.items()})
print("follower inherited deliverable capacity by period:",
      [round(follower_legacy(p), 1) for p in P])
print(f"annualised cost of follower capacity expansion: {CAP_COST:.3f}")

The follower's legacy manufacturing retires in year 22, which is why its inherited capacity drops to
zero in the final two periods. From then on it can only serve the market by paying for expansion —
and that is precisely the moment a leader would want to have pre-committed enough capacity to make
expansion unattractive.

## 3. The MPEC

In [ ]:
NQ = 6          # grid points for the leader's quantity (binary-selected)


def stackelberg(learning='both', mipgap=0.01, env=None, deter=True, nq=NQ):
    """Single-level MPEC.

    The leader's revenue contains -B*qF*qL, a product of two decision variables.
    We keep it EXACT by restricting the leader's quantity to a finite grid chosen by
    binaries: qL = sum_k S_k * bq_k with sum_k bq_k = 1. Then qF*qL = sum_k S_k*(qF*bq_k),
    and each qF*bq_k is a continuous-times-binary product, which linearises exactly.
    deter=False drops the follower entirely (leader as monopolist)."""
    m = gp.Model(env=env) if env is not None else gp.Model()
    m.Params.OutputFlag = 0
    m.Params.MIPGap = mipgap

    L = add_region(m, LEADER, learning)
    qL = L['sale']
    c_f = follower_marginal_cost()
    KQ = list(range(nq))

    # leader quantity on a binary-selected grid
    bq = m.addVars(REGIONS, P, KQ, vtype=GRB.BINARY, name='bq')
    GRID = {}
    for rt in REGIONS:
        for p in P:
            smax = A_INT[rt, p] / B_SLP[rt, p]
            GRID[rt, p] = [smax * k / (nq - 1) for k in KQ]
            m.addConstr(bq.sum(rt, p, '*') == 1, name=f'gsel_{rt}_{p}')
            m.addConstr(qL[rt, p] == gp.quicksum(GRID[rt, p][k] * bq[rt, p, k] for k in KQ),
                        name=f'gq_{rt}_{p}')

    if deter:
        qF = m.addVars(REGIONS, P, lb=0.0, ub=BIG_Q, name='qF')
        Cap = m.addVar(lb=0.0, ub=BIG_Q, name='CapF')
        lam = m.addVars(P, lb=0.0, name='lam')
        nu = m.addVars(REGIONS, P, lb=0.0, name='nu')
        mcap = m.addVar(lb=0.0, name='mcap')
        yc = m.addVars(P, vtype=GRB.BINARY, name='yc')
        zq = m.addVars(REGIONS, P, vtype=GRB.BINARY, name='zq')
        ycap = m.addVar(vtype=GRB.BINARY, name='ycap')
        slack = m.addVars(P, lb=0.0, name='slk')

        # follower primal feasibility
        m.addConstrs((qF.sum('*', p) + slack[p] == follower_legacy(p) + Cap for p in P),
                     name='fcap')
        # follower stationarity
        m.addConstrs((OMEGA[p]*(A_INT[rt, p] - B_SLP[rt, p]*(2*qF[rt, p] + qL[rt, p])
                                - c_f[rt]) - lam[p] + nu[rt, p] == 0
                      for rt in REGIONS for p in P), name='stat_q')
        m.addConstr(-CAP_COST + gp.quicksum(lam[p] for p in P) + mcap == 0, name='stat_cap')
        # complementarity (big-M)
        m.addConstrs((lam[p] <= BIG_L*yc[p] for p in P), name='cc1')
        m.addConstrs((slack[p] <= BIG_Q*(1-yc[p]) for p in P), name='cc2')
        m.addConstrs((nu[rt, p] <= BIG_L*zq[rt, p] for rt in REGIONS for p in P), name='cc3')
        m.addConstrs((qF[rt, p] <= BIG_Q*(1-zq[rt, p]) for rt in REGIONS for p in P), name='cc4')
        m.addConstr(mcap <= BIG_L*ycap, name='cc5')
        m.addConstr(Cap <= BIG_Q*(1-ycap), name='cc6')

        # exact linearisation of w[rt,p,k] = qF[rt,p] * bq[rt,p,k]
        w = m.addVars(REGIONS, P, KQ, lb=0.0, name='w')
        m.addConstrs((w[rt, p, k] <= BIG_Q*bq[rt, p, k]
                      for rt in REGIONS for p in P for k in KQ), name='w1')
        m.addConstrs((w[rt, p, k] <= qF[rt, p]
                      for rt in REGIONS for p in P for k in KQ), name='w2')
        m.addConstrs((w[rt, p, k] >= qF[rt, p] - BIG_Q*(1-bq[rt, p, k])
                      for rt in REGIONS for p in P for k in KQ), name='w3')
    else:
        qF, Cap = None, None

    # leader revenue = A*qL - B*qL^2 - B*qF*qL, all linear on the grid
    rev = gp.LinExpr()
    for rt in REGIONS:
        for p in P:
            for k in KQ:
                Sk = GRID[rt, p][k]
                rev += OMEGA[p]*(A_INT[rt, p]*Sk - B_SLP[rt, p]*Sk*Sk)*bq[rt, p, k]
                if deter:
                    rev -= OMEGA[p]*B_SLP[rt, p]*Sk*w[rt, p, k]

    m.setObjective(rev - L['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._L, m._qL, m._qF, m._Cap, m._rev = L, qL, qF, Cap, rev
    return m

## 4. Validation — does the KKT block reproduce the follower's true optimum?

An MPEC is easy to get subtly wrong: a sign error in stationarity, a big-M that is too small and
silently forces a dual to zero, a missing complementarity pair. The model will still solve and
return plausible numbers.

The check: take the leader's committed quantities out of the MPEC solution, hand them to the
follower's problem solved **directly as a QP**, and compare. If the embedded KKT block is correct,
the two must agree exactly.

In [ ]:
def follower_qp(qL_fixed, env=None):
    """Solve the follower's problem DIRECTLY as a QP, given the leader's quantities.
    Used to verify that the KKT block embedded in the MPEC reproduces it."""
    m = gp.Model(env=env) if env is not None else gp.Model()
    m.Params.OutputFlag = 0
    c_f = follower_marginal_cost()
    qF = m.addVars(REGIONS, P, lb=0.0, ub=BIG_Q, name='qF')
    Cap = m.addVar(lb=0.0, ub=BIG_Q, name='Cap')
    m.addConstrs((qF.sum('*', p) <= follower_legacy(p) + Cap for p in P), name='cap')
    obj = gp.QuadExpr()
    for rt in REGIONS:
        for p in P:
            obj += OMEGA[p]*((A_INT[rt, p] - B_SLP[rt, p]*qL_fixed[rt, p] - c_f[rt])*qF[rt, p]
                             - B_SLP[rt, p]*qF[rt, p]*qF[rt, p])
    obj -= CAP_COST*Cap
    m.setObjective(obj, GRB.MAXIMIZE)
    m.optimize()
    return m, qF, Cap

In [ ]:
m0 = solve_planner(0.5, learning='capacity')
top = {r: m0._H[r]['cum'][P[-1]].X for r in REGIONS}
set_tiers(top)

mpec = stackelberg(learning='both', env=ENV)
mpec.update()
print(f"MPEC: {mpec.NumVars} vars, {mpec.NumConstrs} constraints, "
      f"{mpec.NumBinVars} binaries, status {mpec.Status}")

qL_fix = {(rt, p): mpec._qL[rt, p].X for rt in REGIONS for p in P}
fq, qF_chk, Cap_chk = follower_qp(qL_fix, env=ENV)
print(f"\n  MPEC embedded KKT : follower qty "
      f"{sum(mpec._qF[rt,p].X for rt in REGIONS for p in P):9.4f}  Cap {mpec._Cap.X:8.4f}")
print(f"  direct QP solve   : follower qty "
      f"{sum(qF_chk[rt,p].X for rt in REGIONS for p in P):9.4f}  Cap {Cap_chk.X:8.4f}")
print(f"  max deviation per market-period: "
      f"{max(abs(mpec._qF[rt,p].X - qF_chk[rt,p].X) for rt in REGIONS for p in P):.6f}")

Agreement to machine precision. The KKT block is a faithful representation of the follower's
problem, so the leader really is optimising against a best-responding rival rather than against an
artefact of a mis-specified constraint set.

**Always run this check on an MPEC.** It costs one small QP and it is the only thing standing
between you and a plausible wrong answer.

## 5. Three market structures

- **Monopoly** — no follower at all (`deter=False`). An upper bound on leader profit.
- **Stackelberg** — leader commits, follower best-responds.
- **Cournot** — simultaneous moves, from Part 4c.

In [ ]:
mono = stackelberg(learning='both', deter=False, env=ENV)
cr = cournot_iterate(first='R1', max_iter=16)
lc = {L['firm']: L for L in cr['log'][-2:]}
qL_stack = sum(mpec._qL[rt, p].X for rt in REGIONS for p in P)
qF_stack = sum(mpec._qF[rt, p].X for rt in REGIONS for p in P)

pd.DataFrame([
    dict(structure='Monopoly (no rival)', leader_profit=round(mono.ObjVal, 1),
         leader_qty=round(sum(mono._qL[rt, p].X for rt in REGIONS for p in P), 1),
         follower_qty=0.0, total_qty=round(sum(mono._qL[rt, p].X
                                               for rt in REGIONS for p in P), 1)),
    dict(structure='Stackelberg (leader R1)', leader_profit=round(mpec.ObjVal, 1),
         leader_qty=round(qL_stack, 1), follower_qty=round(qF_stack, 1),
         total_qty=round(qL_stack + qF_stack, 1)),
    dict(structure='Cournot (simultaneous)', leader_profit=round(lc['R1']['profit'], 1),
         leader_qty=round(lc['R1']['sales'], 1), follower_qty=round(lc['R2']['sales'], 1),
         total_qty=round(lc['R1']['sales'] + lc['R2']['sales'], 1)),
])

The ordering is exactly what theory predicts, which is reassuring given how much machinery sits
underneath:

$$\Pi^{\text{monopoly}} \;>\; \Pi^{\text{Stackelberg}} \;>\; \Pi^{\text{Cournot}}$$

**Commitment is worth about +20% of profit to the leader** (13,790 vs 11,527). And the mechanism is
visible in the quantities: the Stackelberg leader produces **more** than it would under Cournot
(1,469 vs 1,267) while the follower produces **less** (719 vs 971).

That is the entire strategic logic. By committing to a large quantity first, the leader moves the
follower down its own reaction curve — every unit the leader commits makes an additional follower
unit less profitable. Overproduction is not a mistake here; it is the instrument.

This is the formal version of the dynamic described at the outset: flood the market to keep a rival
from establishing itself. Note the leader reaches the *monopoly* quantity (1,469 in both rows) but
not monopoly profit, because the follower still supplies 719 units and depresses the price.

## 6. Entry deterrence

The follower's capacity expansion $\kappa$ is the entry decision. How much does the leader's
commitment suppress it?

In [ ]:
rows = []
for tag, mm in [('Stackelberg (leader commits)', mpec)]:
    rows.append(dict(case=tag, follower_expansion=round(mm._Cap.X, 2),
                     follower_qty=round(sum(mm._qF[rt, p].X
                                            for rt in REGIONS for p in P), 1)))
# what would the follower do against a leader that behaved as in Cournot?
qL_cournot = {(rt, p): cr['sales']['R1'][rt, p] for rt in REGIONS for p in P}
fq2, qF2, Cap2 = follower_qp(qL_cournot, env=ENV)
rows.append(dict(case='vs a Cournot-quantity leader',
                 follower_expansion=round(Cap2.X, 2),
                 follower_qty=round(sum(qF2[rt, p].X for rt in REGIONS for p in P), 1)))
pd.DataFrame(rows)

Read the `follower_expansion` column: this is how much new capacity the entrant chooses to build.
Facing a committed leader it invests **less** than it would against a leader playing the Cournot
quantity. The leader's commitment does not merely take market share in the current period — it
suppresses the rival's *capital formation*, which is the durable form of deterrence and the one that
matters for industrial policy.

This is also where the Part 3b learning machinery closes the loop. Less follower output means slower
accumulation of follower production experience, which means the follower stays on a higher
operating-cost tier, which makes future expansion less attractive still. Deterrence compounds.

## 7. How fine does the leader's quantity grid need to be?

The one approximation in this formulation is the discretised leader strategy. Sweep it.

In [ ]:
rows = []
for nq in [3, 4, 6, 8]:
    mm = stackelberg(learning='both', nq=nq, env=ENV)
    if mm.SolCount == 0:
        rows.append(dict(grid_points=nq, status='no solution')); continue
    rows.append(dict(grid_points=nq, leader_profit=round(mm.ObjVal, 1),
                     leader_qty=round(sum(mm._qL[rt, p].X for rt in REGIONS for p in P), 1),
                     follower_qty=round(sum(mm._qF[rt, p].X for rt in REGIONS for p in P), 1),
                     binaries=mm.NumBinVars))
pd.DataFrame(rows)

Coarser grids restrict the leader's strategy space, so they can only **understate** its profit — the
leader is being denied quantities it might prefer. Watch whether the qualitative conclusion (leader
produces above its Cournot quantity, follower's expansion is suppressed) is stable across the sweep.
If it flips between grids, it is a discretisation artefact rather than a result.

The binary count rises quickly: each grid point adds one binary per market-period, on top of the
complementarity binaries and the leader's own build decisions.

## 8. Summary

| Question | Answer |
|---|---|
| Can a bilevel program be solved directly? | No — replace the inner problem with its KKT conditions |
| What makes KKT valid here? | The follower's problem is **continuous and concave** |
| Why was that available? | Part 3 kept the operational layer an LP; integers only on investment |
| How is the bilinear term handled? | Leader quantity on a binary grid → continuous × binary → **exact** |
| Is the embedded KKT correct? | Verified against a direct QP solve, agreement to machine precision |
| What is commitment worth? | ≈ +20% of leader profit over Cournot |
| Does it deter entry? | Yes — the follower's capacity expansion falls |

### Limitations, stated plainly

- **The follower cannot make lumpy investments.** Its capacity is a continuous scalar. Restoring
  binaries there would break KKT and require a different method entirely.
- **Big-M values in the complementarity constraints are chosen, not derived.** Too small silently
  forces duals to zero and yields a wrong answer that still solves; too large destroys the
  relaxation. They deserve the same scrutiny as any other big-M.
- **MPECs violate standard constraint qualifications** at every feasible point — the complementarity
  system has no strict interior. The big-M reformulation sidesteps this by turning it into a MILP,
  which is why that route is standard rather than elegant.
- **Only one leader is modelled.** Which firm leads is assumed, not derived. Solving both directions
  and comparing tells you what leadership is worth, not who gets it.

### What remains: 4e

Policy instruments as exogenous, swept levers — tariffs as arc-cost adders, quotas as arc bounds,
local content minimums (already built in Part 3b). Government constrains, firms respond. With the
MPEC in hand, the natural question is whether a tariff can restore the follower's incentive to
invest in the face of a committed leader.

### Things to try

- Swap leader and follower — set `FOLLOWER, LEADER = 'R1', 'R2'` and re-run
- `CAP_COST` down 30% — cheaper entry, and deterrence should weaken
- `learning='capacity'` — remove the production channel and see how much of the deterrence effect
  was the learning feedback
- Tighten the big-Ms (`BIG_Q`, `BIG_L`) toward their true bounds and watch solve time fall